# 04 - 完整优化实践

## 学习目标

1. 掌握完整的训练循环（loss + optimizer + scheduler）
2. 对比不同优化器和调度器的组合效果
3. 掌握梯度累积技术
4. 掌握梯度裁剪技术
5. 了解混合精度训练的基础用法

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR, OneCycleLR
from torch.utils.data import DataLoader, TensorDataset
import math

## 1. 准备数据和模型

我们使用一个合成的分类数据集来演示完整的优化流程。

In [ ]:
# 生成合成分类数据
torch.manual_seed(42)

n_samples = 2000
n_features = 20
n_classes = 5

# 生成数据
X = torch.randn(n_samples, n_features)
# 使用随机权重生成标签
true_w = torch.randn(n_features, n_classes)
logits = X @ true_w + 0.5 * torch.randn(n_samples, n_classes)
y = logits.argmax(dim=1)

# 划分训练集和验证集
n_train = 1600
X_train, X_val = X[:n_train], X[n_train:]
y_train, y_val = y[:n_train], y[n_train:]

# 创建 DataLoader
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

print(f"训练集: {len(train_dataset)} 样本")
print(f"验证集: {len(val_dataset)} 样本")
print(f"特征维度: {n_features}")
print(f"类别数: {n_classes}")
print(f"每个 epoch 的 batch 数: {len(train_loader)}")

In [ ]:
# 定义模型
class SimpleClassifier(nn.Module):
    """简单的三层分类器。"""
    
    def __init__(self, n_features, n_hidden, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(n_hidden, n_classes),
        )
    
    def forward(self, x):
        return self.net(x)

# 测试模型
model = SimpleClassifier(n_features, 64, n_classes)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(model)

## 2. 完整训练循环

将损失函数、优化器、调度器组合成一个完整的训练流程。

```
for epoch in range(epochs):
    # 训练阶段
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    
    # 验证阶段
    model.eval()
    with torch.no_grad():
        val_loss = evaluate(model, val_loader)
    
    # 更新学习率
    scheduler.step()
```

In [ ]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, scheduler=None, 
                    scheduler_per_batch=False):
    """训练一个 epoch。

    Args:
        model: 模型
        train_loader: 训练数据
        loss_fn: 损失函数
        optimizer: 优化器
        scheduler: 学习率调度器（可选）
        scheduler_per_batch: 是否每个 batch 更新 lr（OneCycleLR 需要）

    Returns:
        平均训练损失
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for x_batch, y_batch in train_loader:
        # 1. 清零梯度
        optimizer.zero_grad()
        # 2. 前向传播
        output = model(x_batch)
        # 3. 计算损失
        loss = loss_fn(output, y_batch)
        # 4. 反向传播
        loss.backward()
        # 5. 更新参数
        optimizer.step()
        
        # 每个 batch 更新 lr（如 OneCycleLR）
        if scheduler_per_batch and scheduler is not None:
            scheduler.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, val_loader, loss_fn):
    """验证模型。

    Args:
        model: 模型
        val_loader: 验证数据
        loss_fn: 损失函数

    Returns:
        (平均验证损失, 准确率)
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for x_batch, y_batch in val_loader:
        output = model(x_batch)
        loss = loss_fn(output, y_batch)
        total_loss += loss.item()
        
        pred = output.argmax(dim=1)
        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)
    
    avg_loss = total_loss / len(val_loader)
    accuracy = correct / total
    return avg_loss, accuracy

print("训练和验证函数已定义")

In [ ]:
# 完整训练循环示例
torch.manual_seed(42)
model = SimpleClassifier(n_features, 64, n_classes)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)

n_epochs = 30
best_val_acc = 0.0

print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'Val Acc':>10} {'LR':>12}")
print('-' * 54)

for epoch in range(n_epochs):
    # 训练
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
    
    # 验证
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    
    # 更新学习率
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    # 记录最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
    
    # 每 5 个 epoch 打印一次
    if epoch % 5 == 0 or epoch == n_epochs - 1:
        print(f"{epoch:6d} {train_loss:12.4f} {val_loss:12.4f} {val_acc:10.4f} {current_lr:12.6f}")

print(f"\n最佳验证准确率: {best_val_acc:.4f}")

## 3. 比较不同优化器 + 调度器组合

对比几种常见的优化组合策略。

In [ ]:
def train_with_config(config_name, optimizer_fn, scheduler_fn, n_epochs=30):
    """使用指定配置训练模型。

    Args:
        config_name: 配置名称
        optimizer_fn: 接受 model.parameters() 返回 optimizer 的函数
        scheduler_fn: 接受 optimizer 返回 scheduler 的函数
        n_epochs: 训练轮数

    Returns:
        (训练 loss 列表, 验证 loss 列表, 验证 acc 列表)
    """
    torch.manual_seed(42)
    model = SimpleClassifier(n_features, 64, n_classes)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optimizer_fn(model.parameters())
    scheduler = scheduler_fn(optimizer) if scheduler_fn else None
    
    train_losses = []
    val_losses = []
    val_accs = []
    
    for epoch in range(n_epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        
        if scheduler is not None:
            scheduler.step()
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
    
    return train_losses, val_losses, val_accs

# 定义几种组合
configs = {
    'SGD+StepLR': (
        lambda p: optim.SGD(p, lr=0.01, momentum=0.9),
        lambda opt: StepLR(opt, step_size=10, gamma=0.5),
    ),
    'Adam(无scheduler)': (
        lambda p: optim.Adam(p, lr=0.001),
        None,
    ),
    'AdamW+Cosine': (
        lambda p: optim.AdamW(p, lr=0.001, weight_decay=0.01),
        lambda opt: CosineAnnealingLR(opt, T_max=30, eta_min=1e-5),
    ),
    'SGD+Cosine': (
        lambda p: optim.SGD(p, lr=0.01, momentum=0.9, weight_decay=1e-4),
        lambda opt: CosineAnnealingLR(opt, T_max=30, eta_min=1e-5),
    ),
}

# 训练所有组合
results = {}
for name, (opt_fn, sched_fn) in configs.items():
    train_losses, val_losses, val_accs = train_with_config(
        name, opt_fn, sched_fn
    )
    results[name] = (train_losses, val_losses, val_accs)

print("训练完成!\n")

In [ ]:
# 打印对比结果
print(f"{'配置':<25} {'最终 Train Loss':>15} {'最终 Val Loss':>15} {'最佳 Val Acc':>12}")
print('-' * 70)
for name, (train_losses, val_losses, val_accs) in results.items():
    print(f"{name:<25} {train_losses[-1]:>15.4f} {val_losses[-1]:>15.4f} {max(val_accs):>12.4f}")

print("\n每 10 个 epoch 的验证准确率:")
print(f"{'Epoch':>6}", end='')
for name in results:
    print(f"  {name:>20}", end='')
print()
print('-' * (6 + 22 * len(results)))
for epoch in [0, 9, 19, 29]:
    print(f"{epoch:6d}", end='')
    for name in results:
        print(f"  {results[name][2][epoch]:>20.4f}", end='')
    print()

## 4. 梯度累积（Gradient Accumulation）

当 GPU 显存不足以放下大 batch 时，可以用梯度累积来模拟大 batch 训练。

**核心思想**：多个小 batch 的梯度累加后再更新参数。

```
有效 batch_size = 实际 batch_size x 累积步数
```

In [ ]:
# 梯度累积示例
def train_with_gradient_accumulation(model, train_loader, loss_fn, optimizer, 
                                     accumulation_steps=4):
    """带梯度累积的训练。

    Args:
        model: 模型
        train_loader: 训练数据
        loss_fn: 损失函数
        optimizer: 优化器
        accumulation_steps: 梯度累积步数

    Returns:
        平均训练损失
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    optimizer.zero_grad()  # 初始化梯度为零
    
    for i, (x_batch, y_batch) in enumerate(train_loader):
        # 前向传播
        output = model(x_batch)
        # 损失除以累积步数（保持梯度量级一致）
        loss = loss_fn(output, y_batch) / accumulation_steps
        # 反向传播（梯度会累加）
        loss.backward()
        
        total_loss += loss.item() * accumulation_steps
        n_batches += 1
        
        # 每累积够 accumulation_steps 步，更新一次参数
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
    
    # 处理最后不足 accumulation_steps 的批次
    if (i + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()
    
    return total_loss / n_batches

# 对比：普通训练 vs 梯度累积
torch.manual_seed(42)
model_normal = SimpleClassifier(n_features, 64, n_classes)
opt_normal = optim.AdamW(model_normal.parameters(), lr=0.001)

torch.manual_seed(42)
model_accum = SimpleClassifier(n_features, 64, n_classes)
opt_accum = optim.AdamW(model_accum.parameters(), lr=0.001)

loss_fn = nn.CrossEntropyLoss()

print("梯度累积对比 (accumulation_steps=4):")
print(f"  有效 batch_size: 64 x 4 = 256\n")

for epoch in range(5):
    loss_normal = train_one_epoch(model_normal, train_loader, loss_fn, opt_normal)
    loss_accum = train_with_gradient_accumulation(
        model_accum, train_loader, loss_fn, opt_accum, accumulation_steps=4
    )
    
    _, acc_normal = evaluate(model_normal, val_loader, loss_fn)
    _, acc_accum = evaluate(model_accum, val_loader, loss_fn)
    
    print(f"  Epoch {epoch}: 普通 loss={loss_normal:.4f} acc={acc_normal:.4f} | "
          f"累积 loss={loss_accum:.4f} acc={acc_accum:.4f}")

print("\n说明: 梯度累积等效于更大的 batch size")
print("  适用场景: 显存不足、需要大 batch 训练（如 NLP 任务）")

## 5. 梯度裁剪（Gradient Clipping）

防止梯度爆炸，常用于 RNN/Transformer 训练。

两种方式：
- `clip_grad_norm_`：裁剪梯度的总范数
- `clip_grad_value_`：裁剪梯度的每个元素值

In [ ]:
# 梯度裁剪示例
torch.manual_seed(42)
model = SimpleClassifier(n_features, 64, n_classes)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# 一个训练步骤
x, y_target = next(iter(train_loader))
output = model(x)
loss = loss_fn(output, y_target)
loss.backward()

# 查看裁剪前的梯度范数
total_norm_before = torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=float('inf')  # 不裁剪，只计算范数
)
print(f"裁剪前梯度总范数: {total_norm_before:.4f}")

# 重新计算梯度
optimizer.zero_grad()
loss = loss_fn(model(x), y_target)
loss.backward()

# 方法 1: clip_grad_norm_（裁剪梯度总范数，最常用）
max_norm = 1.0
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
print(f"\nclip_grad_norm_ (max_norm={max_norm}):")
print(f"  裁剪后梯度总范数: {total_norm:.4f}")

# 重新计算梯度
optimizer.zero_grad()
loss = loss_fn(model(x), y_target)
loss.backward()

# 方法 2: clip_grad_value_（裁剪每个元素的值）
clip_value = 0.5
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=clip_value)
print(f"\nclip_grad_value_ (clip_value={clip_value}):")
# 验证所有梯度值都在 [-clip_value, clip_value] 范围内
all_clipped = True
for p in model.parameters():
    if p.grad is not None:
        if p.grad.abs().max() > clip_value + 1e-6:
            all_clipped = False
print(f"  所有梯度值在 [-{clip_value}, {clip_value}] 范围内: {all_clipped}")

In [ ]:
# 带梯度裁剪的完整训练循环
def train_with_gradient_clipping(model, train_loader, loss_fn, optimizer, max_norm=1.0):
    """带梯度裁剪的训练。

    Args:
        model: 模型
        train_loader: 训练数据
        loss_fn: 损失函数
        optimizer: 优化器
        max_norm: 最大梯度范数

    Returns:
        (平均损失, 平均梯度范数)
    """
    model.train()
    total_loss = 0.0
    total_grad_norm = 0.0
    n_batches = 0
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        
        # 梯度裁剪（在 optimizer.step() 之前！）
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        optimizer.step()
        
        total_loss += loss.item()
        total_grad_norm += grad_norm.item()
        n_batches += 1
    
    return total_loss / n_batches, total_grad_norm / n_batches

# 训练并监控梯度范数
torch.manual_seed(42)
model = SimpleClassifier(n_features, 64, n_classes)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

print("带梯度裁剪的训练 (max_norm=1.0):")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Avg Grad Norm':>15}")
print('-' * 35)

for epoch in range(10):
    train_loss, avg_grad_norm = train_with_gradient_clipping(
        model, train_loader, loss_fn, optimizer, max_norm=1.0
    )
    if epoch % 2 == 0:
        print(f"{epoch:6d} {train_loss:12.4f} {avg_grad_norm:15.4f}")

print("\n说明: 梯度裁剪确保梯度范数不超过 max_norm")
print("  顺序: backward() -> clip_grad_norm_() -> step()")

## 6. 混合精度训练基础（torch.amp）

使用 FP16/BF16 加速训练，同时保持 FP32 精度。

**核心组件**：
- `torch.amp.autocast`：自动选择操作的精度
- `torch.amp.GradScaler`：缩放梯度防止 FP16 下溢

**注意**：混合精度训练主要在 GPU 上有加速效果。

In [ ]:
# 混合精度训练示例（CPU 上演示原理，实际加速需要 GPU）

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前设备: {device}")

if device.type == 'cuda':
    # GPU 上的混合精度训练
    torch.manual_seed(42)
    model = SimpleClassifier(n_features, 64, n_classes).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')
    
    print("\n混合精度训练 (GPU):")
    for epoch in range(5):
        model.train()
        total_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            
            # autocast: 自动选择操作精度
            with torch.amp.autocast('cuda'):
                output = model(x_batch)
                loss = loss_fn(output, y_batch)
            
            # 缩放损失并反向传播
            scaler.scale(loss).backward()
            # 先 unscale 梯度，再做梯度裁剪
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # 更新参数
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"  Epoch {epoch}: loss={avg_loss:.4f}")
else:
    print("\n当前无 GPU，展示混合精度训练的代码模式:")
    print("""
    # 混合精度训练模板
    scaler = torch.amp.GradScaler('cuda')
    
    for data, target in train_loader:
        optimizer.zero_grad()
        
        # 1. autocast 上下文管理器
        with torch.amp.autocast('cuda'):
            output = model(data)
            loss = loss_fn(output, target)
        
        # 2. scaler 缩放损失
        scaler.scale(loss).backward()
        
        # 3. unscale 后做梯度裁剪（可选）
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 4. scaler 更新参数
        scaler.step(optimizer)
        scaler.update()
    """)

print("\n混合精度训练的好处:")
print("  1. 减少显存使用（约 50%）")
print("  2. 加速计算（Tensor Cores）")
print("  3. 几乎不损失精度")

In [ ]:
# 混合精度训练中不同操作的精度策略
print("autocast 下不同操作的精度:")
print()
print("FP16 执行的操作（计算密集型）:")
print("  - 矩阵乘法 (Linear, Conv, MatMul)")
print("  - 批归一化 (BatchNorm)")
print()
print("FP32 执行的操作（精度敏感型）:")
print("  - 损失函数 (CrossEntropyLoss, MSELoss)")
print("  - Softmax")
print("  - LayerNorm")
print("  - 指数运算 (exp, log, pow)")
print()
print("GradScaler 的作用:")
print("  - FP16 的数值范围较小，小梯度可能下溢为 0")
print("  - GradScaler 在 backward 前放大 loss（scale up）")
print("  - 在 optimizer.step 前缩小梯度（unscale）")
print("  - 如果检测到 inf/nan，跳过该步更新")

## 7. 练习题

### 练习 1：实现最佳实践的完整训练流程

组合以下技术训练一个模型：
- AdamW 优化器（不同参数组的权重衰减）
- Warmup + Cosine 学习率调度
- 梯度裁剪
- 保存最佳模型 checkpoint

In [ ]:
# 练习 1: 最佳实践训练流程
def train_best_practices(model, train_loader, val_loader, n_epochs=30):
    """使用最佳实践训练模型。

    Args:
        model: 模型
        train_loader: 训练数据
        val_loader: 验证数据
        n_epochs: 训练轮数

    Returns:
        最佳验证准确率
    """
    # TODO: 实现
    # 1. 分离 weight 和 bias 参数
    # 2. 创建 AdamW（bias 不做权重衰减）
    # 3. 创建 warmup + cosine scheduler
    # 4. 训练循环中加入梯度裁剪
    # 5. 保存最佳模型
    pass

# 测试
# torch.manual_seed(42)
# model = SimpleClassifier(n_features, 64, n_classes)
# best_acc = train_best_practices(model, train_loader, val_loader)
# print(f"最佳验证准确率: {best_acc:.4f}")

### 练习 2：实现带梯度累积和裁剪的训练

将梯度累积和梯度裁剪结合在一起。

In [ ]:
# 练习 2: 梯度累积 + 梯度裁剪
def train_advanced(model, train_loader, loss_fn, optimizer,
                   accumulation_steps=4, max_norm=1.0):
    """带梯度累积和裁剪的训练。

    Args:
        model: 模型
        train_loader: 训练数据
        loss_fn: 损失函数
        optimizer: 优化器
        accumulation_steps: 累积步数
        max_norm: 最大梯度范数

    Returns:
        平均训练损失
    """
    # TODO: 实现
    # 注意: 梯度裁剪应在累积完成后、step() 之前进行
    pass

# 测试
# torch.manual_seed(42)
# model = SimpleClassifier(n_features, 64, n_classes)
# optimizer = optim.AdamW(model.parameters(), lr=0.001)
# loss_fn = nn.CrossEntropyLoss()
# loss = train_advanced(model, train_loader, loss_fn, optimizer)
# print(f"训练损失: {loss:.4f}")

## 8. 小结

### 完整训练最佳实践

```python
# 1. 模型
model = MyModel(cfg)

# 2. 损失函数
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# 3. 优化器（分参数组）
optimizer = optim.AdamW([
    {'params': weight_params, 'weight_decay': 0.01},
    {'params': bias_params, 'weight_decay': 0.0},
], lr=1e-3)

# 4. 调度器（warmup + cosine）
scheduler = get_cosine_schedule_with_warmup(...)

# 5. 训练循环
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):        # 混合精度
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        clip_grad_norm_(model.parameters(), 1.0)  # 梯度裁剪
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()                          # 每 batch 更新 lr
    
    # 验证 + 保存最佳模型
    val_loss, val_acc = evaluate(model, val_loader)
    if val_acc > best_acc:
        save_checkpoint(model, optimizer, scheduler, epoch)
```

### 本章回顾

| 技术 | 作用 | 关键点 |
|------|------|--------|
| 损失函数 | 衡量预测与真实的差距 | 分类用 CE，回归用 MSE |
| 优化器 | 根据梯度更新参数 | 默认用 AdamW |
| 学习率调度 | 动态调整学习率 | Warmup + Cosine |
| 梯度累积 | 模拟大 batch | loss 除以累积步数 |
| 梯度裁剪 | 防止梯度爆炸 | max_norm=1.0 |
| 混合精度 | 加速 + 省显存 | autocast + GradScaler |

### 调试技巧

- loss 不下降：检查学习率是否太大或太小
- loss 变 NaN：检查梯度爆炸，加梯度裁剪
- 训练不稳定：加 warmup
- 过拟合：加 weight_decay、dropout、label_smoothing
- 显存不足：减小 batch_size 或用梯度累积